# 🏗️ Setup Inicial: Crear Schemas y Tablas Delta

**Objetivo:** Crear toda la estructura de bases de datos y tablas necesarias para el pipeline ETL.

**⚠️ EJECUTAR ESTE NOTEBOOK ANTES QUE TODOS LOS DEMÁS**

---

## 📋 Lo que hace:
1. Crear databases: `bronze`, `silver`, `gold`
2. Crear tablas Bronze (API responses)
3. Crear tablas Silver (tickers, OHLCV)
4. Crear tablas Gold (features, predictions, portfolio)

---

**Duración:** ~2 minutos  
**Frecuencia:** Solo 1 vez (o cuando resetees el entorno)

## 📦 Step 1: Inicializar Spark

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

print("✅ Spark session iniciada")
print(f"   Versión Spark: {spark.version}")
print(f"   Catalog: {spark.catalog.currentDatabase()}")

## 🗄️ Step 2: Crear Databases

In [0]:
# Crear databases
databases = ['bronze', 'silver', 'gold']

for db in databases:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    print(f"✅ Database '{db}' creada/verificada")

# Mostrar databases existentes
print("\n📊 Databases disponibles:")
spark.sql("SHOW DATABASES").show(truncate=False)

## 🥉 Step 3: Crear Tablas Bronze Layer

In [0]:
# Bronze: API Responses (raw JSON)
spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.api_responses (
    response_id STRING NOT NULL,
    endpoint STRING NOT NULL,
    request_params MAP<STRING, STRING>,
    response_body STRING,
    response_status INT,
    response_headers MAP<STRING, STRING>,
    ingestion_timestamp TIMESTAMP NOT NULL,
    data_date DATE NOT NULL,
    ticker STRING,
    is_successful BOOLEAN NOT NULL,
    error_message STRING
)
USING DELTA
PARTITIONED BY (data_date)
COMMENT 'Raw API responses from DataBursátil - Complete audit trail'
""")

print("✅ Tabla 'bronze.api_responses' creada")

# Verificar
spark.sql("DESCRIBE TABLE bronze.api_responses").show(truncate=False)

## 🥈 Step 4: Crear Tablas Silver Layer

In [0]:
# Silver: Tickers (catálogo maestro)
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.tickers (
    ticker STRING NOT NULL,
    is_active BOOLEAN NOT NULL,
    first_seen TIMESTAMP NOT NULL,
    last_seen TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Master ticker catalog - ticker-agnostic design'
""")

print("✅ Tabla 'silver.tickers' creada")

# Silver: Market OHLCV (normalized data)
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.market_ohlcv (
    ticker STRING NOT NULL,
    date DATE NOT NULL,
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE NOT NULL,
    volume BIGINT,
    adjusted_close DOUBLE,
    source STRING,
    ingestion_timestamp TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (date)
COMMENT 'Normalized OHLCV data in long format - ticker-agnostic'
""")

print("✅ Tabla 'silver.market_ohlcv' creada")

# Verificar tablas silver
print("\n📊 Tablas en schema 'silver':")
spark.sql("SHOW TABLES IN silver").show(truncate=False)

## 🥇 Step 5: Crear Tablas Gold Layer

In [0]:
# Gold: Runs (versioning/tracking)
spark.sql("""
CREATE TABLE IF NOT EXISTS gold.runs (
    run_id STRING NOT NULL,
    run_timestamp TIMESTAMP NOT NULL,
    pipeline_stage STRING NOT NULL,
    data_ref STRING,
    features_version STRING,
    code_version STRING,
    config MAP<STRING, STRING>,
    status STRING NOT NULL,
    error_message STRING
)
USING DELTA
COMMENT 'Central versioning table for reproducibility'
""")

print("✅ Tabla 'gold.runs' creada")

# Gold: Features Daily
spark.sql("""
CREATE TABLE IF NOT EXISTS gold.features_daily (
    ticker STRING NOT NULL,
    date DATE NOT NULL,
    features_version STRING NOT NULL,
    features MAP<STRING, DOUBLE> NOT NULL,
    run_id STRING NOT NULL,
    created_at TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (date)
COMMENT 'Technical indicators and ML features - extensible MAP schema'
""")

print("✅ Tabla 'gold.features_daily' creada")

# Gold: Predictions
spark.sql("""
CREATE TABLE IF NOT EXISTS gold.predictions (
    prediction_id STRING NOT NULL,
    ticker STRING NOT NULL,
    prediction_date DATE NOT NULL,
    target_date DATE NOT NULL,
    model_name STRING NOT NULL,
    model_version STRING NOT NULL,
    predicted_price DOUBLE NOT NULL,
    confidence_interval_lower DOUBLE,
    confidence_interval_upper DOUBLE,
    run_id STRING NOT NULL,
    created_at TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (prediction_date)
COMMENT 'LSTM model predictions'
""")

print("✅ Tabla 'gold.predictions' creada")

# Gold: Portfolio Holdings
spark.sql("""
CREATE TABLE IF NOT EXISTS gold.portfolio_holdings (
    portfolio_id STRING NOT NULL,
    rebalance_date DATE NOT NULL,
    ticker STRING NOT NULL,
    weight DOUBLE NOT NULL,
    shares BIGINT,
    strategy STRING NOT NULL,
    run_id STRING NOT NULL,
    created_at TIMESTAMP NOT NULL
)
USING DELTA
PARTITIONED BY (rebalance_date)
COMMENT 'Optimized portfolio holdings'
""")

print("✅ Tabla 'gold.portfolio_holdings' creada")

# Verificar tablas gold
print("\n📊 Tablas en schema 'gold':")
spark.sql("SHOW TABLES IN gold").show(truncate=False)

## ✅ Step 6: Verificación Final

In [0]:
print("=" * 60)
print("✅ SETUP COMPLETADO")
print("=" * 60)

# Resumen de tablas creadas
tablas_esperadas = {
    'bronze': ['api_responses'],
    'silver': ['tickers', 'market_ohlcv'],
    'gold': ['runs', 'features_daily', 'predictions', 'portfolio_holdings']
}

print("\n📊 Resumen de tablas creadas:\n")
for db, tablas in tablas_esperadas.items():
    print(f"🗄️  {db.upper()} ({len(tablas)} tablas):")
    for tabla in tablas:
        # Verificar que exista
        try:
            count = spark.sql(f"SELECT COUNT(*) as count FROM {db}.{tabla}").collect()[0]['count']
            print(f"   ✅ {db}.{tabla} - {count} registros")
        except Exception as e:
            print(f"   ❌ {db}.{tabla} - ERROR: {str(e)[:50]}")

print("\n" + "=" * 60)
print("🚀 PRÓXIMOS PASOS:")
print("=" * 60)
print("1. ✅ Ejecutar: notebooks/setup/01_populate_ticker_catalog.ipynb")
print("2. ✅ Ejecutar: notebooks/bronze/01_ingest_databursatil.ipynb")
print("3. ✅ Ejecutar: notebooks/silver/02_normalize_to_silver.ipynb")
print("4. ✅ Ejecutar: notebooks/gold/03_feature_engineering.ipynb")
print("=" * 60)

## 📝 Notas Importantes

### ✅ Lo que se creó:

**Bronze Layer:**
- `bronze.api_responses` - Audit trail completo de llamadas API

**Silver Layer:**
- `silver.tickers` - Catálogo maestro de emisoras (vacío, llenar con siguiente notebook)
- `silver.market_ohlcv` - Datos OHLCV normalizados (vacío)

**Gold Layer:**
- `gold.runs` - Tracking de ejecuciones (vacío)
- `gold.features_daily` - Features técnicas (vacío)
- `gold.predictions` - Predicciones LSTM (vacío)
- `gold.portfolio_holdings` - Portafolios optimizados (vacío)

---

### ⚠️ Storage Location

Por defecto, Databricks guarda las tablas en:
```
dbfs:/user/hive/warehouse/<database>.db/<table>/
```

Si necesitas configurar ADLS Gen2, modifica los CREATE TABLE:
```sql
CREATE TABLE silver.tickers (...)
USING DELTA
LOCATION 'abfss://silver@<storage>.dfs.core.windows.net/tickers'
```

---

### 🔄 Reset Completo (si necesitas empezar de cero)

```sql
-- ⚠️ CUIDADO: Esto BORRA TODO
DROP DATABASE IF EXISTS bronze CASCADE;
DROP DATABASE IF EXISTS silver CASCADE;
DROP DATABASE IF EXISTS gold CASCADE;

-- Luego re-ejecuta este notebook
```